In [1]:
import os
import json
import openai

import data_utils

/workspace/Label-free-CBM/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


In [2]:
dataset = "birds525"
prompt_type = "important"

In [3]:
openai.api_key = open(os.path.join(os.path.expanduser("/workspace"), ".openai_api_key"), "r").read()[:-1]

In [4]:
prompts = {
    "important" : "List the most important features for recognizing something as a \"goldfish\":\n\n-bright orange color\n-a small, round body\n-a long, flowing tail\n-a small mouth\n-orange fins\n\nList the most important features for recognizing something as a \"beerglass\":\n\n-a tall, cylindrical shape\n-clear or translucent color\n-opening at the top\n-a sturdy base\n-a handle\n\nList the most important features for recognizing something as a \"{}\":",
    "superclass" : "Give superclasses for the word \"tench\":\n\n-fish\n-vertebrate\n-animal\n\nGive superclasses for the word \"beer glass\":\n\n-glass\n-container\n-object\n\nGive superclasses for the word \"{}\":",
    "around" : "List the things most commonly seen around a \"tench\":\n\n- a pond\n-fish\n-a net\n-a rod\n-a reel\n-a hook\n-bait\n\nList the things most commonly seen around a \"beer glass\":\n\n- beer\n-a bar\n-a coaster\n-a napkin\n-a straw\n-a lime\n-a person\n\nList the things most commonly seen around a \"{}\":"
}

base_prompt = prompts[prompt_type]

In [5]:
cls_file = data_utils.LABEL_FILES[dataset]
with open(cls_file, "r") as f:
    classes = f.read().split("\n")

In [6]:
from openai import OpenAI
import os

key_path = os.path.join(os.path.expanduser("/workspace"), ".openai_api_key")
client = OpenAI(api_key=open(key_path).read().strip())

print(client.models.list().data[0].id)   # auth check

text-embedding-ada-002


In [11]:
# Smoke test

import re, textwrap

PROMPTS = {
    "important":  "List the most important features for recognizing something as a \"{}\":",
    "around":     "List the things most commonly seen around a \"{}\":",
    "superclass": "Give superclasses for the word \"{}\":",
}

SYSTEM = ("You continue a bulleted list of short visual attributes. Each item "
          "must be a generic visual property under 30 characters (e.g. 'long "
          "curved beak', 'black wings'). Never mention the species name. No "
          "preamble, no explanations, no full sentences. Only the list.")

MODEL, MAX_LEN = "gpt-4o-mini", 30
test_classes = classes[:3]

def parse(text):
    out = [re.sub(r"^[-*\u2022]\s*", "", l).strip().strip(".").lower()
           for l in text.split("\n")]
    return [c for c in out if c]

for ptype, tmpl in PROMPTS.items():
    print("=" * 70, f"\n{ptype.upper()}\n" + "=" * 70)
    for cls in test_classes:
        r = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": SYSTEM},
                      {"role": "user", "content": tmpl.format(cls)}],
            temperature=0.3, max_tokens=200)
        raw = r.choices[0].message.content
        cs  = parse(raw)

        kept    = [c for c in cs if len(c) <= MAX_LEN]
        leaked  = [c for c in cs if cls.lower() in c or c in cls.lower()]
        bullets = [c for c in cs if c.startswith(("-", "*"))]
        avg     = sum(len(c) for c in cs) / max(len(cs), 1)

        print(f"\n{cls}")
        print(f"  raw preview : {textwrap.shorten(repr(raw), 90)}")
        print(f"  parsed      : {len(cs)}  avg len {avg:.0f}  "
              f"survive <={MAX_LEN}: {len(kept)}/{len(cs)}")
        if bullets: print(f"  !! BULLETS LEFT : {bullets[:3]}")
        if leaked:  print(f"  !! NAME LEAKED  : {leaked[:3]}")
        print(f"  sample      : {cs[:5]}")

IMPORTANT

abbotts babbler
  raw preview : '- brown streaked plumage \n- short tail \n- pale underparts \n- dark eye stripe \n- [...]
  parsed      : 10  avg len 15  survive <=30: 10/10
  sample      : ['brown streaked plumage', 'short tail', 'pale underparts', 'dark eye stripe', 'slender body']

abbotts booby
  raw preview : '- white head \n- dark brown body \n- long pointed wings \n- slender tail \n- pale [...]
  parsed      : 10  avg len 13  survive <=30: 10/10
  sample      : ['white head', 'dark brown body', 'long pointed wings', 'slender tail', 'pale underbelly']

abyssinian ground hornbill
  raw preview : '- large size \n- long curved beak \n- black feathers \n- red throat pouch \n- white [...]
  parsed      : 10  avg len 15  survive <=30: 10/10
  sample      : ['large size', 'long curved beak', 'black feathers', 'red throat pouch', 'white wing tips']
AROUND

abbotts babbler
  raw preview : '- brown streaked plumage \n- short tail \n- pale underparts \n- dark eye stripe \n- [...

In [7]:
import json, os, re, time
from tqdm.auto import tqdm

PROMPTS = {
    "important":  "List the most important features for recognizing something as a \"{}\":",
    "around":     "List the things most commonly seen around a \"{}\":",
    "superclass": "Give superclasses for the word \"{}\":",
}

SYSTEMS = {
    "important": ("You continue a bulleted list of short visual attributes. Each "
                  "item is a generic visual property under 30 characters (e.g. "
                  "'long curved beak', 'black wings')."),
    "around":    ("You continue a bulleted list of things commonly seen in the "
                  "surroundings or habitat of the subject — objects, scenery, and "
                  "environment, NOT features of the animal itself. Each item is "
                  "under 30 characters (e.g. 'shallow water', 'tree branch', "
                  "'rocky cliff', 'tall grass')."),
    "superclass": ("You continue a bulleted list of broader categories the subject "
                   "belongs to. Each item is a general noun under 30 characters "
                   "(e.g. 'bird', 'seabird', 'animal', 'wading bird'). Output at "
                   "most 5 items."),
}

COMMON = (" Never mention the species name. No preamble, no explanations, no full "
          "sentences. Output only the list, one item per line.")

MODEL, N_SAMPLES, TEMP = "gpt-4o-mini", 2, 0.3


def parse(text):
    out = [re.sub(r"^[-*\u2022]\s*", "", l).strip().strip(".").lower()
           for l in text.split("\n")]
    return [c for c in out if c]


def ask(cls, ptype):
    for attempt in range(6):
        try:
            r = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "system", "content": SYSTEMS[ptype] + COMMON},
                          {"role": "user", "content": PROMPTS[ptype].format(cls)}],
                temperature=TEMP, max_tokens=256, top_p=1,
                frequency_penalty=0, presence_penalty=0)
            return parse(r.choices[0].message.content)
        except Exception as e:
            tqdm.write(f"    retry {attempt+1} ({type(e).__name__})")
            time.sleep(2 ** attempt)
    tqdm.write(f"    GIVING UP: {ptype}/{cls}")
    return []


assert len(classes) == 525, f"expected 525 classes, got {len(classes)}"
os.makedirs("data/concept_sets/gpt3_init", exist_ok=True)

for ptype in PROMPTS:
    cache_path = f"data/gpt_cache_{dataset}_{ptype}.json"
    feature_dict = json.load(open(cache_path)) if os.path.exists(cache_path) else {}
    todo = [c for c in classes if c not in feature_dict]

    if not todo:
        tqdm.write(f"{ptype}: already complete ({len(feature_dict)} classes)")
    else:
        pbar = tqdm(todo, desc=f"{ptype:<11}", unit="cls")
        for i, label in enumerate(pbar):
            feats = set()
            for _ in range(N_SAMPLES):
                feats.update(ask(label, ptype))
            feature_dict[label] = sorted(feats)

            pbar.set_postfix_str(f"{label[:22]} ({len(feats)})")
            if i % 25 == 0:
                json.dump(feature_dict, open(cache_path, "w"))

        json.dump(feature_dict, open(cache_path, "w"))

    out = f"data/concept_sets/gpt3_init/gpt3_{dataset}_{ptype}.json"
    json.dump(feature_dict, open(out, "w"), indent=2)
    n_concepts = sum(len(v) for v in feature_dict.values())
    tqdm.write(f"{ptype}: {len(feature_dict)} classes, "
               f"{n_concepts} concepts -> {out}")

important: already complete (525 classes)
important: 525 classes, 5698 concepts -> data/concept_sets/gpt3_init/gpt3_birds525_important.json
around: already complete (525 classes)
around: 525 classes, 8145 concepts -> data/concept_sets/gpt3_init/gpt3_birds525_around.json
superclass: already complete (525 classes)
superclass: 525 classes, 2625 concepts -> data/concept_sets/gpt3_init/gpt3_birds525_superclass.json


In [9]:
json_object = json.dumps(feature_dict, indent=4)
with open("data/concept_sets/gpt3_init/gpt3_{}_{}_new.json".format(dataset, prompt_type), "w") as outfile:
    outfile.write(json_object)